In [ ]:
from google.colab import drive
import pandas as pd
import glob
import joblib

# 1. Connect to Google Drive
drive.mount('/content/drive')

# 2. Load the trained Random Forest model directly from Drive
model_path = '/content/drive/MyDrive/EncryptedSDP/notebooks/random_forest_aligned_28f.pkl'
rf_model = joblib.load(model_path)
print("SUCCESS: Trained Random Forest model loaded from Drive.\n")

# 3. Define paths for the independent BCCC testing dataset
# (Make sure these match where you saved the BCCC testing files in Phase 2)
test_benign_path = "/content/drive/MyDrive/EncryptedSDP/datasets/testing/raw/benign-csvs/*.csv"
test_malic_path = "/content/drive/MyDrive/EncryptedSDP/datasets/testing/raw/malic-csvs/*.csv"

print("Ready to load the unseen BCCC testing data.")

In [ ]:
import os

print("Checking the contents of your datasets folder...")
base_path = "/content/drive/MyDrive/EncryptedSDP/datasets/"

if os.path.exists(base_path):
    print("\nFolders inside 'datasets':")
    print(os.listdir(base_path))
else:
    print("\nCould not find the 'datasets' folder at that path.")

In [ ]:
import pandas as pd
import glob

print("Locating the BCCC testing dataset...\n")

# 1. Find the single CSV file in either the raw or extracted folder
search_paths = [
    "/content/drive/MyDrive/EncryptedSDP/datasets/testing/raw/*.csv",
    "/content/drive/MyDrive/EncryptedSDP/datasets/testing/extracted/*.csv"
]

bccc_file = None
for path in search_paths:
    files = glob.glob(path)
    if files:
        bccc_file = files[0]
        break

if bccc_file:
    print(f"SUCCESS: Found test file at -> {bccc_file}")

    # 2. Load the dataset
    print("Loading data into memory (this might take a moment)...")
    full_test_df = pd.read_csv(bccc_file)

    # 3. Print the columns and labels to verify the structure
    print(f"\nTotal Rows in Test Data: {len(full_test_df):,}")
    print("\n--- LAST 5 COLUMNS ---")
    print(full_test_df.columns[-5:].tolist())

    # Check the exact name of the label column (usually 'Label' or 'class')
    label_col = 'Label' if 'Label' in full_test_df.columns else full_test_df.columns[-1]
    print(f"\n--- LABEL DISTRIBUTION IN '{label_col}' ---")
    print(full_test_df[label_col].value_counts())

else:
    print("ERROR: Could not find the CSV file. Please check the folder paths.")

In [ ]:
print("--- ALL BCCC TEST FEATURES ---")
for i, col in enumerate(full_test_df.columns, 1):
    print(f"{i}. {col}")

In [ ]:
import joblib

# Load the trained 28-feature Random Forest model
model_path = '/content/drive/MyDrive/EncryptedSDP/notebooks/random_forest_aligned_28f.pkl'

rf_model_aligned = joblib.load(model_path)

print("28-feature Random Forest model loaded successfully!")

In [ ]:
import numpy as np
import pandas as pd

# Make a copy of the testing dataset
test_df = full_test_df.copy()

# Separate labels from input features
y_test = test_df['Label'].map({
    'Benign': 0,
    'Malicious': 1
})

X_test = test_df.drop(columns=['Label'])

# Match the trained model's feature order
X_test = X_test[rf_model_aligned.feature_names_in_]

# Convert features to numeric values
X_test = X_test.apply(pd.to_numeric, errors='coerce')

# Replace infinite values with missing values
X_test = X_test.replace([np.inf, -np.inf], np.nan)

# Keep only valid records
valid_rows = X_test.notna().all(axis=1) & y_test.notna()

X_test = X_test.loc[valid_rows]
y_test = y_test.loc[valid_rows].astype(int)

print("Original testing records:", len(full_test_df))
print("Records removed:", len(full_test_df) - len(X_test))
print("Final testing records:", len(X_test))
print("Features:", X_test.shape[1])

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import joblib

print("Loading the aligned 28-feature Random Forest model...\n")
aligned_model_path = '/content/drive/MyDrive/EncryptedSDP/notebooks/random_forest_aligned_28f.pkl'
rf_model_aligned = joblib.load(aligned_model_path)

print("Deploying model on unseen BCCC network traffic...")

# 1. Predict the labels for the unseen BCCC features
y_pred_bccc = rf_model_aligned.predict(X_test)

# 2. Generate the Final Real-World Metrics
print("\n--- FINAL INDEPENDENT BCCC TEST METRICS ---")
print(classification_report(y_test, y_pred_bccc, target_names=['Benign (0)', 'Malicious (1)']))

# 3. Generate the Final Visual Confusion Matrix
cm = confusion_matrix(y_test, y_pred_bccc)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malicious'],
            yticklabels=['Benign', 'Malicious'])
plt.ylabel('Actual True Label')
plt.xlabel('Model Predicted Label')
plt.title('Final Confusion Matrix - Unseen BCCC Data (28 Features)')
plt.show()

In [ ]:
import os
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

print("--- EXPORTING FINAL RESULTS TO GOOGLE DRIVE ---")

# Define where you want to save the files
save_folder = '/content/drive/MyDrive/EncryptedSDP/notebooks/'

# 1. Export the Text Metrics to a .txt file
report_text = classification_report(y_test, y_pred_bccc, target_names=['Benign (0)', 'Malicious (1)'])
report_path = os.path.join(save_folder, 'bccc_test_results.txt')

with open(report_path, 'w') as f:
    f.write("--- FINAL INDEPENDENT BCCC TEST METRICS ---\n\n")
    f.write(report_text)
print(f"SUCCESS: Text metrics saved to -> {report_path}")

# 2. Export the Confusion Matrix to a high-res .png graphic
cm = confusion_matrix(y_test, y_pred_bccc)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malicious'],
            yticklabels=['Benign', 'Malicious'])
plt.ylabel('Actual True Label')
plt.xlabel('Model Predicted Label')
plt.title('Final Confusion Matrix - Unseen BCCC Data (28 Features)')

# Save the figure as a PNG before displaying it
matrix_path = os.path.join(save_folder, 'bccc_confusion_matrix.png')
plt.savefig(matrix_path, bbox_inches='tight', dpi=300) # dpi=300 makes it crisp for slides
print(f"SUCCESS: Confusion matrix graphic saved to -> {matrix_path}\n")

# Display the plot
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

print("--- EXTRACTING FEATURE IMPORTANCE ---")

# 1. Extract the importance scores from the aligned model
importances = rf_model_aligned.feature_importances_
feature_names = X_test.columns

# 2. Create a DataFrame and sort them from most to least important
feature_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

print("\nTop 5 Most Critical Features for Detecting Encrypted Malware:")
print(feature_df.head(5).to_string(index=False))

# 3. Plot the Top 10 Features
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_df.head(10), palette='viridis')
plt.title('Top 10 Most Important Features for DoH Malware Detection', fontsize=14)
plt.xlabel('Relative Importance Score', fontsize=12)
plt.ylabel('Network Flow Feature', fontsize=12)

# 4. Save the graphic to your Drive
save_folder = '/content/drive/MyDrive/EncryptedSDP/notebooks/'
importance_path = os.path.join(save_folder, 'feature_importance_chart.png')
plt.savefig(importance_path, bbox_inches='tight', dpi=300)
print(f"\nSUCCESS: Feature Importance chart saved to -> {importance_path}\n")

# Display the plot
plt.show()